In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/zomato.db')
print("Connected to zomato.db")

def run_query(sql):
    return pd.read_sql_query(sql, conn)

Connected to zomato.db


In [6]:
q1=run_query("""
    SELECT
        location,
        COUNT(*) AS total_rataurants,
        ROUND(AVG(rate)) AS avg_rating,
        SUM(votes) AS total_votes,
        ROUND(AVG(cost_for_two),0) AS avg_cost
    FROM restaurants
    WHERE rate IS NOT NULL
    GROUP BY location
    HAVING COUNT(*)>=20
    ORDER BY avg_rating DESC, total_votes DESC
    LIMIT 20;   
""")
q1

,location,total_rataurants,avg_rating,total_votes,avg_cost
0,Koramangala 5th Block,2319,4.0,2219506,678.0
1,Indiranagar,1847,4.0,1196007,672.0
2,Koramangala 4th Block,841,4.0,685156,758.0
3,BTM,3930,4.0,617880,419.0
4,Church Street,546,4.0,594979,840.0
5,JP Nagar,1717,4.0,578071,554.0
6,Lavelle Road,487,4.0,506186,1353.0
7,HSR,2019,4.0,499720,499.0
8,Koramangala 7th Block,1060,4.0,495324,603.0
9,Jayanagar,1643,4.0,481285,500.0


In [8]:
q2=run_query("""
    SELECT
             primary_cuisine,
             COUNT(*) AS restaurant_count,
             ROUND(AVG(rate),2) AS avg_rating,
             Round(AVG(cost_for_two),0) AS avg_cost,
             SUM(votes) AS total_votes,
             ROUND(
                CAST(SUM(votes)AS FLOAT)/
                (SELECT SUM(votes) FROM restaurants)*100 ,2) AS vote_share_percent
    FROM restaurants
    WHERE primary_cuisine!='Unknown'
    AND rate IS NOT NULL
    GROUP BY primary_cuisine
    HAVING COUNT(*)>=15
    ORDER BY total_votes DESC
    LIMIT 20;

""")
q2

,primary_cuisine,restaurant_count,avg_rating,avg_cost,total_votes,vote_share_percent
0,North Indian,9840,3.59,635.0,3274711,22.32
1,Cafe,3987,3.87,637.0,1880756,12.82
2,Continental,1671,3.98,1141.0,1612050,10.99
3,American,441,4.16,1353.0,840749,5.73
4,Chinese,2539,3.65,648.0,712822,4.86
5,Biryani,2227,3.53,451.0,565478,3.85
6,Pizza,586,3.74,612.0,468772,3.20
7,Italian,625,3.93,1071.0,446248,3.04
8,South Indian,3546,3.59,350.0,433864,2.96
9,Finger Food,659,3.88,1436.0,391771,2.67


In [13]:
q3 = run_query("""
    SELECT
             CASE WHEN online_order=1 THEN 'Online' ELSE 'Offline' END AS order_mode,
             COUNT(*) AS restaurant_count,
             ROUND(AVG(rate),2) AS avg_rating,
             ROUND(AVG(cost_for_two))AS avg_cost,
             SUM(votes)AS total_votes
    FROM restaurants
    WHERE rate IS NOT NULL
    GROUP BY online_order
    ORDER BY avg_rating DESC;
""")    
q3

,order_mode,restaurant_count,avg_rating,avg_cost,total_votes
0,Online,27206,3.72,544.0,9337879
1,Offline,14459,3.66,711.0,5313744


In [ ]:
q4 = run_query("""
    SELECT 
        rest_type,
        COUNT(*) AS total_restaurants,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(AVG(cost_for_two), 0) AS avg_cost,
        ROUND(AVG(votes), 0) AS avg_votes,
        ROUND(
            AVG(rate) / (AVG(cost_for_two) / 100), 3) AS value_score
    FROM restaurants
    WHERE rest_type != 'Unknown'
    AND rate IS NOT NULL
    GROUP BY rest_type
    HAVING COUNT(*) >= 30
    ORDER BY avg_rating DESC
    LIMIT 10
""")

q4

,rest_type,total_restaurants,avg_rating,avg_cost,avg_votes,value_score
0,"Pub, Cafe",40,4.66,1285.0,4473.0,0.362
1,"Pub, Microbrewery",76,4.45,1814.0,4614.0,0.245
2,"Microbrewery, Pub",42,4.44,1693.0,5504.0,0.262
3,"Microbrewery, Casual Dining",121,4.37,1676.0,1299.0,0.261
4,"Casual Dining, Microbrewery",59,4.31,1283.0,2527.0,0.336
5,"Fine Dining, Bar",40,4.21,3003.0,434.0,0.140
6,"Casual Dining, Cafe",318,4.19,908.0,1663.0,0.462
7,"Casual Dining, Pub",127,4.18,1320.0,1418.0,0.316
8,"Lounge, Casual Dining",37,4.15,1741.0,1885.0,0.238
9,Fine Dining,343,4.15,2721.0,490.0,0.153


In [15]:
q5 = run_query("""
    WITH ranked AS (
        SELECT 
            name,
            location,
            rate,
            votes,
            cost_for_two,
            primary_cuisine,
            ROW_NUMBER() OVER (
                PARTITION BY location
                ORDER BY rate DESC, votes DESC
                ) AS area_rank
        FROM restaurants
        WHERE rate IS NOT NULL 
        AND votes >= 50
    )
    SELECT * FROM ranked
    WHERE area_rank <= 3
    ORDER BY location, area_rank
    LIMIT 20
""")

q5

,name,location,rate,votes,cost_for_two,primary_cuisine,area_rank
0,AB's - Absolute Barbecues,BTM,4.9,6490,1600.0,European,1
1,AB's - Absolute Barbecues,BTM,4.9,6452,1600.0,European,2
2,AB's - Absolute Barbecues,BTM,4.9,6404,1600.0,European,3
3,Taaza Thindi,Banashankari,4.7,651,100.0,South Indian,1
4,Onesta,Banashankari,4.6,2604,600.0,Pizza,2
5,Onesta,Banashankari,4.6,2556,600.0,Pizza,3
6,Corner House Ice Cream,Banaswadi,4.4,220,400.0,Ice Cream,1
7,Saffron Vegetarian Cuisine,Banaswadi,4.0,671,700.0,North Indian,2
8,Saffron Vegetarian Cuisine,Banaswadi,4.0,667,700.0,North Indian,3
9,Galito's,Bannerghatta Road,4.6,415,1000.0,African,1


In [16]:
q6 = run_query("""
    WITH location_stats AS (
        SELECT
            location,
            COUNT(*) AS restaurant_count,
            SUM(votes) AS total_demand,
            ROUND(AVG(rate), 2) AS avg_rating,
            ROUND(AVG(cost_for_two), 0) AS avg_spend
        FROM restaurants
        GROUP BY location
    ),
    scored AS (
        SELECT *,
            ROUND(CAST(total_demand AS FLOAT) / 
                NULLIF(restaurant_count, 0), 0) AS demand_per_restaurant
        FROM location_stats
    )
    SELECT
        location,
        restaurant_count,
        total_demand,
        demand_per_restaurant,
        avg_rating,
        avg_spend,
        CASE
            WHEN demand_per_restaurant > 500 
                AND restaurant_count < 100 THEN 'High Opportunity'
            WHEN demand_per_restaurant > 300 
                AND restaurant_count < 150 THEN 'Growing Area'
            ELSE 'Established'
        END AS expansion_signal
    FROM scored
    ORDER BY demand_per_restaurant DESC
    LIMIT 20
""")

q6

,location,restaurant_count,total_demand,demand_per_restaurant,avg_rating,avg_spend,expansion_signal
0,Church Street,569,594979,1046.0,3.99,835.0,Established
1,Lavelle Road,529,506186,957.0,4.14,1298.0,Established
2,Koramangala 5th Block,2504,2219506,886.0,4.01,661.0,Established
3,St. Marks Road,352,266099,756.0,4.02,871.0,Established
4,Koramangala 4th Block,1017,685156,674.0,3.92,696.0,Established
5,Cunningham Road,491,287873,586.0,3.90,865.0,Established
6,Koramangala 3rd Block,216,125159,579.0,4.02,778.0,Established
7,Indiranagar,2083,1196007,574.0,3.83,648.0,Established
8,MG Road,918,432111,471.0,3.86,1136.0,Established
9,Residency Road,675,291954,433.0,3.86,965.0,Established


In [17]:
q7 = run_query("""
    WITH area_stats AS (
        SELECT
            location,
            ROUND(AVG(cost_for_two), 0) AS avg_cost,
            ROUND(AVG(rate), 2) AS avg_rating,
            COUNT(*) AS total_restaurants,
            NTILE(4) OVER (ORDER BY AVG(cost_for_two)) AS cost_quartile
        FROM restaurants
        WHERE rate IS NOT NULL
        AND cost_for_two IS NOT NULL
        GROUP BY location
        HAVING COUNT(*) >= 20
    )
    SELECT
        location,
        avg_cost,
        avg_rating,
        total_restaurants,
        CASE cost_quartile
            WHEN 1 THEN 'Budget'
            WHEN 2 THEN 'Mid Range'
            WHEN 3 THEN 'Premium'
            WHEN 4 THEN 'Luxury'
        END AS price_segment
    FROM area_stats
    ORDER BY avg_cost DESC
    LIMIT 15
""")

q7

,location,avg_cost,avg_rating,total_restaurants,price_segment
0,Sankey Road,2583.0,3.97,26,Luxury
1,Lavelle Road,1353.0,4.14,487,Luxury
2,Race Course Road,1321.0,3.78,135,Luxury
3,MG Road,1226.0,3.86,811,Luxury
4,Infantry Road,1073.0,3.80,140,Luxury
5,Residency Road,1029.0,3.86,605,Luxury
6,Richmond Road,897.0,3.82,612,Luxury
7,St. Marks Road,884.0,4.02,343,Luxury
8,Langford Town,883.0,3.81,27,Luxury
9,Cunningham Road,867.0,3.90,475,Luxury


In [18]:
q8 = run_query("""
    SELECT
        CASE WHEN book_table = 1 THEN 'Accepts Booking' 
             ELSE 'No Booking' END AS booking_status,
        COUNT(*) AS restaurant_count,
        ROUND(AVG(rate), 2) AS avg_rating,
        ROUND(AVG(cost_for_two), 0) AS avg_cost,
        ROUND(AVG(votes), 0) AS avg_votes,
        SUM(votes) AS total_votes
    FROM restaurants
    WHERE rate IS NOT NULL
    GROUP BY book_table
    ORDER BY avg_rating DESC
""")

q8

,booking_status,restaurant_count,avg_rating,avg_cost,avg_votes,total_votes
0,Accepts Booking,6304,4.14,1276.0,1171.0,7384146
1,No Booking,35361,3.62,482.0,206.0,7267477


In [19]:
q9 = run_query("""
    WITH cuisine_area AS (
        SELECT
            location,
            primary_cuisine,
            COUNT(*) AS restaurant_count,
            ROUND(AVG(rate), 2) AS avg_rating,
            SUM(votes) AS total_votes,
            ROW_NUMBER() OVER (
                PARTITION BY location
                ORDER BY SUM(votes) DESC
            ) AS cuisine_rank
        FROM restaurants
        WHERE rate IS NOT NULL
        AND primary_cuisine != 'Unknown'
        GROUP BY location, primary_cuisine
        HAVING COUNT(*) >= 5
    )
    SELECT
        location,
        primary_cuisine AS top_cuisine,
        restaurant_count,
        avg_rating,
        total_votes
    FROM cuisine_area
    WHERE cuisine_rank = 1
    ORDER BY total_votes DESC
    LIMIT 15
""")

q9

,location,top_cuisine,restaurant_count,avg_rating,total_votes
0,Koramangala 5th Block,Cafe,410,4.14,520290
1,Indiranagar,North Indian,327,3.68,240465
2,Sarjapur Road,Continental,58,4.18,201339
3,Jayanagar,North Indian,323,3.78,193735
4,Cunningham Road,North Indian,155,4.07,190550
5,BTM,North Indian,1200,3.46,189339
6,Marathahalli,North Indian,534,3.49,186418
7,Koramangala 4th Block,Pizza,27,4.15,174547
8,Church Street,Cafe,124,4.09,158677
9,JP Nagar,North Indian,389,3.61,150472


In [20]:
q10 = run_query("""
    WITH engagement AS (
        SELECT
            location,
            COUNT(*) AS total_restaurants,
            SUM(votes) AS total_votes,
            ROUND(AVG(votes), 0) AS avg_votes_per_restaurant,
            ROUND(AVG(rate), 2) AS avg_rating,
            RANK() OVER (
                ORDER BY AVG(votes) DESC
            ) AS engagement_rank
        FROM restaurants
        WHERE rate IS NOT NULL
        GROUP BY location
        HAVING COUNT(*) >= 20
    )
    SELECT
        location,
        total_restaurants,
        total_votes,
        avg_votes_per_restaurant,
        avg_rating,
        engagement_rank
    FROM engagement
    WHERE engagement_rank <= 15
    ORDER BY engagement_rank
""")

q10

,location,total_restaurants,total_votes,avg_votes_per_restaurant,avg_rating,engagement_rank
0,Church Street,546,594979,1090.0,3.99,1
1,Lavelle Road,487,506186,1039.0,4.14,2
2,Koramangala 5th Block,2319,2219506,957.0,4.01,3
3,Koramangala 4th Block,841,685156,815.0,3.92,4
4,St. Marks Road,343,266099,776.0,4.02,5
5,Koramangala 3rd Block,191,125159,655.0,4.02,6
6,Indiranagar,1847,1196007,648.0,3.83,7
7,Cunningham Road,475,287873,606.0,3.90,8
8,MG Road,811,432111,533.0,3.86,9
9,Residency Road,605,291954,483.0,3.86,10
